# 07 — Model Comparison and Evaluation**Step 9 of the brief.** Three models, three tasks, three very different outcomes — and thecomparison is where the actual learning is.

In [ ]:
import sys, pathlibsys.path.append(str(pathlib.Path.cwd().parent))   # [so `from src...` works from notebooks/]import numpy as np, pandas as pdimport matplotlib.pyplot as plt, seaborn as snsfrom src.config import *sns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (9, 5)pd.set_option("display.max_columns", 40)

In [ ]:
import jsonclf = json.load(open(RESULT_DIR / "classification_results.json"))reg = json.load(open(RESULT_DIR / "regression_results.json"))clu = json.load(open(RESULT_DIR / "clustering_results.json"))summary = pd.DataFrame([    ["Clarity classifier", "ANN (8-class)", "Accuracy",   f"{clf['accuracy']:.3f}",     f"baseline {clf['baseline']:.3f}"],    ["Clarity classifier", "ANN (8-class)", "Macro F1",   f"{clf['f1_macro']:.3f}",     "every class counts equally"],    ["Clarity classifier", "ANN (8-class)", "Within 1 grade", f"{clf['within_1_grade']:.3f}",     "the number that is actually useful"],    ["Price regressor",    "ANN (linear out)", "R2",      f"{reg['r2']:.4f}",     "0 = guessing the mean"],    ["Price regressor",    "ANN (linear out)", "MAE",     f"${reg['mae']:,.0f}",     "average miss in dollars"],    ["Price regressor",    "ANN (linear out)", "RMSE",    f"${reg['rmse']:,.0f}",     "> MAE, so a few large misses exist"],    ["Price regressor",    "ANN (linear out)", "MAPE",    f"{reg['mape']:.1f}%",     "what the log transform optimised"],    ["Segmentation",       "K-Means",       "Silhouette", f"{clu['silhouette']:.3f}",     f"k = {int(clu['k'])}"],], columns=["Model", "Type", "Metric", "Score", "Note"])display(summary)summary.to_csv(RESULT_DIR / "model_comparison.csv", index=False)

## The finding that matters**Same architecture. Same effort. Same data. Wildly different results.**The price regressor reaches R² ≈ 0.98. The clarity classifier roughly doubles its baseline andthen stalls in the mid-50s. Neither outcome is about the model.**Price is *physically determined* by the features we have.** Carat alone correlates above 0.9with it. The information is genuinely present in the inputs, and a competent network extracts it.**Clarity is not.** Clarity is a judgment about **microscopic inclusions inside the crystal** —flaws a human grader sees under 10× magnification. Nothing in carat, depth, table, x, y, z, cutor colour *measures* those. We are inferring an optical property from geometry and price, andthe ceiling on that inference is low no matter how many layers we stack.> **A model's quality is bounded by the information in its features. No amount of architecture> search conjures signal that isn't in the data.**This is the single most valuable thing in the whole project, and it is the thing that separatessomeone who can *run* scikit-learn from someone who can *do* machine learning. When a modelunderperforms, the first question is never "which optimiser?" — it is **"is the answer even inthe inputs?"**

## Strengths and limitations### Clarity classifier (ANN, 8-class)**Strengths** — beats the majority-class baseline by roughly 2×. Errors cluster on *adjacent*grades, so the "within 1 grade" figure is high and genuinely actionable: it's good enough to**flag certificates that look wrong** for human re-inspection, which is a real business use.Class weighting stopped it abandoning the rare grades.**Limitations** — the features cannot see inclusions, so the ceiling is structural. Severe classimbalance (I1 is under 1.5% of rows). And `price` is in the feature set, which is defensible forsanity-checking a listed stone but makes it useless for grading an *unpriced* one — be explicitabout that scope in the README rather than letting a reader assume otherwise.### Price regressor (ANN, linear output)**Strengths** — R² ≈ 0.98, MAPE in the single digits. The log-price transform is what made thiswork: it converts the loss from "minimise squared dollars" (which over-weights the rareexpensive stones by ~100×) into "minimise proportional error", which is what a jeweller actuallycares about. The flat percentage error across price bands is the evidence it worked.**Limitations** — trained on one snapshot of one market. It has no concept of a fluctuatingdiamond price index, of brand, of certification body, or of the fluorescence and polish gradesa real appraisal uses. It will extrapolate badly outside the carat range it saw. **And agradient-boosted tree would very likely match or beat it here on tabular data with far lesstuning** — the ANN is the assignment's requirement, not the objectively best tool for this shapeof problem. Say so; it demonstrates judgment.### Segmentation (K-Means)**Strengths** — produces interpretable tiers a merchandiser can act on. Silhouette confirms thechosen k. It independently rediscovered the carat–clarity confound we spotted in the EDA, whichis a nice validation that the structure is real and not an artefact of our feature choices.**Limitations** — K-Means *always* returns k clusters whether or not k clusters exist, and thisdata is a **continuous cloud**, not naturally separated blobs. The segments are therefore**imposed** rather than **discovered** — a useful business abstraction, not a law of nature. Behonest about that distinction. DBSCAN, tried in notebook 05, correctly fails on this data forexactly that reason: there are no density gaps to find.

## Improvements and optimisation strategies**Feature side (highest expected return, and this is not a coincidence)**- The clarity model's ceiling is a *data* problem, not a model problem. The fix is more  informative inputs: inclusion counts, fluorescence, polish and symmetry grades, certification  body. Nothing else will move it much.- Interaction terms — `carat × clarity_rank` — to let the model express the confound explicitly.- Target-encode the grades by their mean price within carat bands.**Model side**- Compare against **XGBoost / LightGBM**. On tabular data they usually beat a plain ANN, train  in seconds, and hand you feature importances for free. Reporting that a simpler model wins is  a *finding*, not a failure — the goal is a working system, not a validated preference.- Keras Tuner or Optuna over layer widths, dropout rate and learning rate, rather than  hand-picking them as we did.- 5-fold cross-validation instead of a single split, so the reported score comes with an error  bar rather than pretending to be exact.- Ensemble several seeds and average — cheap variance reduction.**Evaluation side**- Cost-sensitive metrics. Misgrading an I1 as an IF is a commercially serious error;  misgrading VS1 as VS2 is not. Plain accuracy treats them identically, which is wrong.- Prediction *intervals* on price, not just point estimates. "\$4,200 ± \$600" is a far more  honest and more usable output for an appraiser than "\$4,200".**Deployment side**- Pickle the scaler *with* the model, always. A model without its scaler is a loaded gun with no  sights — it will run, and it will be silently, confidently wrong.- Monitor for drift. Diamond prices move; a model trained on one snapshot decays.